# 7.1. From Fully Connected Layers to Convolutions
D2L의 From Fully Connected Layers to Convolutions장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 완전연결층에서 합성곱으로

지금까지 배운 MLP는 일반적인 데이터에서는 잘 사용할 수 있다. 

예를 들어서 표 형태의 데이터라면
- 나이
- 키
- 몸무게
- 소득

이런 feature를 입력으로 사용할 수 있다. 하지만 이미지는 일반적인 표 데이터와 다르다. 이미지는 단순히 수 많은 숫자의 모음이 아니라 픽셀들이 공간적으로 배치되어 있는 데이터이다. `CNN(Convolutional Neural Network)`은 이러한 이미지의 공간 구조를 활용하도록 만들어진 신경망이다.


## 2. 이미지에 MLP를 사용한다면?

이미지가 가로 1000픽셀에 세로 1000픽셀이라고 생각해보면

    1000 x 1000 = 1,000,000 pixels

이미지를 MLP에 넣으려면 먼저 펼쳐야한다.

[1000, 1000] -> Flatten -> [1,000,000]

그리고 은닉 뉴런 1000개를 사용한다고 해보면

nn.Linear(1000000, 1000) 이다.

필요한 weight 개수는 1,000,000 x 1000 = 1,000,000,000 약 10억개이다.

단순히 계산량만 큰 것이 아니고 이미지를 Flatten하면 어떤 픽셀이 어떤 픽셀 옆에 있는지? 같은 이미지 공간 구조도 제대로 활용하지 못한다.

In [1]:
input_features = 1000 * 1000
hidden_features = 1000

num_weights = input_features * hidden_features

print(f"{num_weights:,}")

1,000,000,000


## 3. 이미지가 일반 데이터와 다른 이유

이미지에는 중요한 공간적 특징이 있다. 예를 들어서 고양이 귀를 찾는다고 생각했을 때

고양이 귀가 왼쪽 중앙 오른쪽 어느 위치에 있든 고양이 귀이다. 어떤 픽셀을 이해하기 위해서 이미지 전체의 모든 픽셀을 처음부터 볼 필요도 없다.

보통 가까이 있는 픽셀들이 서로 더 강하게 관련되어 있다. CNN은 이러한 이미지의 특성을 이용한다. D2L에선 CNN 설계 원칙을 크게 translation 관련 성질과 locality로 정리하고, 깊은 층으로 갈수록 더 넓은 범위의 특징을 표현해야 한다고 설명합니다.

## 4. Translation Equivariance (위치가 바뀌어도 같은 특징)

이미지 안에서 같은 물체가 위치만 바뀌었다고 생각해보자.

예를 들어서 고양이 귀가 왼쪽에 있을 때도 오른쪽에 있을 때도 같은 귀 탐지 방법을 사용할 수 있어야 한다. 

위치마다 서로 다은 detector를 만들 필요가 없다. 하나의 detector를 이미지 전체에서 반복해 사용할 수 있다.

이것이 CNN의 핵심 아이디어 중 하나이다. 같은 kernel(filter)을 이미지 전체 위치에서 공유한다.

Convolution 자체는 엄밀히 말하면 translation invariant보단 translation equivariant에 가깝다.

### Invariance와 Equivariance 차이

Translation Equivariance: 입력이 이동하면 출력 feature map도 함께 이동한다.

입력:
고양이 귀가 오른쪽으로 이동

출력:
귀를 감지한 위치도 오른쪽으로 이동

Translation Invariance: 입력 위치가 바뀌어도 최종 출력은 동일하다.

예:

왼쪽에 고양이 -> "고양이"
오른쪽에 고양이 -> "고양이"

Convolution은 기본적으로 Translation Equivariance를 제공하고,
Pooling이나 최종 분류 과정 등을 거치면서
위치 변화에 더 강한 표현을 만들 수 있다.

## 5. Locality (가까운 영역부터 보기)

어떤 픽셀 주변에서 특징을 찾을 때 이미지 전체를 볼 필요는 없다. 

예를 들어서 눈 가장자리를 찾을 때

현재 위치 주변의

3 x 3
5 x 5

정도의 작은 영역만 봐도 선이나 경계 같은 특징을 찾을 수 있다. 이것을 Locality(지역성) 이라고 한다.

CNN은 하나의 뉴런이 이미지 전체를 보는 대신 작은 영역만 보도록 만든다.

## 6. MLP를 제한하면 CNN이 된다.

일반적인 완전연결층에서는

모든 입력 픽셀 -> 모든 출력 뉴런

이 서로 연결된다. 하지만 이미지에선 두 가지 조건을 적용할 수 있다.

### 조건 1

위치가 달라도 같은 weight를 사용한다.

-> Weight Sharing


### 조건 2

현재 위치 주변의 작은 영역만 본다.

-> Locality


이 두 조건을 MLP에 적용하면 Convolution Layer가 만들어진다.

```text
Fully Connected

모든 위치 × 모든 위치
        ↓

Weight Sharing
        +
Local Connectivity
        ↓

Convolution
```

## 7. kernel(filter)이란?

CNN에선 작은 weight 행렬을 이미지 위에서 움직이며 사용해야한다.

이 작은 weight 행렬을 

- Kernel
- Filter

라고 부른다. 예를 들어서 3 x 3 kernel이라면

k = [ k11  k12  k13 ]
    [ k21  k22  k23 ]
    [ k31  k32  k33 ]

이다. 이 kernel을 이미지의 여러 위치에서 반복해 적용한다.

각 위치마다 새로운 weight를 만드는 것은 아니고 같은 kernel의 weight를 재사용한다. 이것을 weight sharing이라고 한다.


## 8. 합성곱 연산의 직관

이미지의 작은 영역과 kernel을 가져온다.

Image Patch

[ x11  x12  x13 ]
[ x21  x22  x23 ]
[ x31  x32  x33 ]

Kernel

[ w11  w12  w13 ]
[ w21  w22  w23 ]
[ w31  w32  w33 ]

같은 위치의 값을 곱한 뒤 모두 더한다.

output = x11*w11 + x12*w12 + x13*w13 + x21*w21 + x22*w22 + x23*w23 + x31*w31 + x32*w32 + x33*w33

이 결과 하나가 output feature map의 한 위치가 된다. kernel을 이동하며 같은 연산을 반복한다.

## 9. CNN의 파라미터가 적은 이유

MLP에서는 입력마다 각각 다른 weight가 필요하다. CNN에서는 작은 kernel 하나를 이미지 전체에서 공유한다.

예를 들어서 grayscale 이미지에

3 × 3 kernel 하나를 사용한다면 필요한 weight는

3 × 3 = 9개

뿐이다.

이미지가 28 × 28이든 100 × 100이든 1000 × 1000이든

같은 3 × 3 kernel을 사용할 수 있다. CNN의 파라미터 수는 이미지의 전체 픽셀 개수에 직접 비례하지 않는다.

이게 CNN이 이미지를 효율적으로 처리할 수 있는 중요한 이유이다.